# Deliverable 1: Casen 2024 & Censo 2024 (Corregido)

## 1. Importación y Configuración

In [ ]:
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("viridis")


## 2. Lectura CENSO 2024

In [ ]:
censo_viv_path = "../data/raw/viviendas_censo2024.parquet"
df_censo_viv = pl.read_parquet(censo_viv_path).filter(pl.col("cant_per") > 0)
display(df_censo_viv.head(3).to_pandas())


## 3. Lectura CASEN 2024

In [ ]:
casen_path = "../data/raw/casen_2024.dta"
Col_variables = ['region', 'expr', 'v15', 'ind_hacina', 'v3', 'pobreza_multi']
df_casen = pd.read_stata(casen_path, columns=Col_variables).dropna(subset=['region', 'v15'])
display(df_casen.head(3))


## 4. Agregación Multi-Indicador

In [ ]:
# Aseguramos que la tabla no venga vacía y usamos la Casen nativamente para no fallar cruces
casen_reg = df_casen.groupby("region", observed=False).apply(lambda x: pd.Series({
    "%_Subsidio": (x["v15"].astype(str).str.contains("Sí|Si", case=False, na=False) * x["expr"]).sum() / x["expr"].sum() * 100,
    "%_Pobreza": (x["pobreza_multi"].astype(str).str.contains("Pobre|pobre", case=False, na=False) * x["expr"]).sum() / x["expr"].sum() * 100,
    "%_Hacinamiento": (x["ind_hacina"].astype(str).str.contains("hacinamiento", case=False, na=False) * x["expr"]).sum() / x["expr"].sum() * 100
})).reset_index()

casen_reg.set_index("region", inplace=True)
casen_reg = casen_reg.dropna() # Evitar nulos

# Normalizar columnas para el heatmap (escala relativa 0 a 1 min-max)
df_plot1_norm = (casen_reg - casen_reg.min()) / (casen_reg.max() - casen_reg.min())

print("Visualizando los indicadores agregados por región:")
display(casen_reg.round(2))


## 5. Gráfico 1: Heatmap (CASEN)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(df_plot1_norm, cmap='coolwarm', annot=casen_reg.round(1), fmt='.1f', 
            linewidths=1, ax=ax, cbar_kws={'label': 'Intensidad de Problemática (0 a 1)'})

ax.set_title("1. Intensidad de Pobreza, Subsidio y Hacinamiento", fontsize=13, fontweight='bold')
ax.set_ylabel("Regiones")
ax.set_xlabel("Indicadores Relativos")
plt.xticks(rotation=15)
plt.show()


## 6. Gráfico 2: Calidad Muros (CASEN)

In [ ]:
t_sub = np.where(df_casen['v15'].astype(str).str.contains("Sí|Si", case=False, na=False), 'Con Subsidio', 'Sin Subsidio')
t_pob = np.where(df_casen['pobreza_multi'].astype(str).str.contains("Pobre|pobre", case=False, na=False), 'Pobre', 'No Pobre')

ct_g2 = pd.crosstab([t_pob, t_sub], df_casen['v3'], normalize='index') * 100

fig, ax = plt.subplots(figsize=(7, 5))
ct_g2.plot(kind='bar', stacked=True, ax=ax, colormap='RdYlGn_r')
ax.set_title("2. Calidad de Muros vs. Sector Socioeconómico", fontsize=13, fontweight='bold')
ax.set_ylabel("% de Hogares")
ax.set_xlabel("Sector Socioeconómico")
ax.legend(title="Conservación", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.show()


## 7. Gráfico 3: Cohabitantes vs Dormitorios (CENSO)

In [ ]:
df_plot3 = df_censo_viv.filter((pl.col("p5_num_dormitorios") <= 6) & (pl.col("cant_per") <= 10)).select(["p5_num_dormitorios", "cant_per"]).to_pandas()
ct_g3 = pd.crosstab(df_plot3['cant_per'], df_plot3['p5_num_dormitorios'])

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(ct_g3, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Cantidad de Viviendas Censo'})
ax.invert_yaxis()
ax.set_title("3. Frecuencia de Cohabitantes vs Dormitorios Disponibles", fontsize=13, fontweight='bold')
ax.set_xlabel("Número de Dormitorios (Declarados)")
ax.set_ylabel("Personas Viviendo Juntas")
plt.show()


## 8. Gráfico 4: Distribución (CASEN)

In [ ]:
temp_g4 = pd.DataFrame({
    'Subsidio': np.where(df_casen['v15'].astype(str).str.contains("Sí|Si", case=False, na=False), 'Con Subsidio', 'Sin Subsidio'),
    'Hacinamiento': df_casen['ind_hacina']
})

fig, ax = plt.subplots(figsize=(7, 5))
sns.countplot(data=temp_g4, x='Subsidio', hue='Hacinamiento', ax=ax, palette='OrRd')
ax.set_title("4. Carga de Hacinamiento por Subsidio Habitacional", fontsize=13, fontweight='bold')
ax.set_xlabel("Estado de Subsidio")
ax.set_ylabel("Hogares Encuestados")
ax.legend(title="Índice", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()


## 9. Exportación A4 Final

In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(11.69, 8.27))
plt.subplots_adjust(hspace=0.45, wspace=0.55, top=0.88, left=0.08, right=0.92, bottom=0.1)
fig.suptitle("Deliverable 1: Paradoja del Subsidio Habitacional (2024)", fontsize=15, fontweight='bold')

sns.heatmap(df_plot1_norm, cmap='coolwarm', annot=casen_reg.round(1), fmt='.1f', linewidths=1, ax=axs[0,0], cbar=False)
axs[0,0].set_title("1. Pobreza/Subsidio/Hacinamiento (Región)", fontsize=10, fontweight='bold')
axs[0,0].tick_params(axis='x', rotation=15)
axs[0,0].set_ylabel("")

ct_g2.plot(kind='bar', stacked=True, ax=axs[0,1], colormap='RdYlGn_r', legend=False)
axs[0,1].set_title("2. Conservación Muros vs Entorno", fontsize=10, fontweight='bold')
axs[0,1].tick_params(axis='x', rotation=45)

sns.heatmap(ct_g3, cmap='YlOrRd', ax=axs[1,0], cbar=False)
axs[1,0].invert_yaxis()
axs[1,0].set_title("3. Habitantes vs Dormitorios (Censo)", fontsize=10, fontweight='bold')

sns.countplot(data=temp_g4, x='Subsidio', hue='Hacinamiento', ax=axs[1,1], palette='OrRd')
axs[1,1].set_title("4. Carga de Hacinamiento Temporal", fontsize=10, fontweight='bold')
axs[1,1].legend(title="Hacinamiento", bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)

fig.text(0.5, 0.02, "Autor: [TU NOMBRE] | Fuentes: CASEN 2024 y CENSO 2024", ha='center', fontsize=9, color='gray')
plt.savefig("../deliverables/Deliverable_1_Layout.pdf", format="pdf", bbox_inches="tight", dpi=300)
plt.close(fig)
print("¡Archivo PDF generado exitosamente!")
